# Tesla Autopilot Clone: Object Detection

**YOLOv8 object detection on the KITTI dataset — a Kaggle-ready, end-to-end pipeline**

## 1. Project Overview

**Goal.** Build a computer-vision system inspired by autonomous-driving perception that detects road users in front-camera images: cars, vans, trucks, trams, pedestrians, cyclists, seated persons and miscellaneous objects.

**Use case.** Better object detection is a building block for safer driver-assistance and autonomous-driving systems (this project is an educational *clone-style* study, not Tesla's actual stack).

**Pipeline implemented in this notebook**

1. Automatically discover the KITTI object-detection dataset under `/kaggle/input/` (no hard-coded folder names).
2. Explore the data (structure, image sizes, class distribution, objects per image, annotated samples).
3. Convert KITTI annotations (`x1, y1, x2, y2` in pixels) to YOLO format (`class x_center y_center width height`, normalised to 0–1).
4. Verify the conversion numerically and visually.
5. Create a reproducible train / validation / test split (fixed seed) and `data.yaml`.
6. Fine-tune a pretrained lightweight **YOLOv8n** with Ultralytics.
7. Evaluate `best.pt` on validation **and** test splits: precision, recall, mAP@0.5, mAP@0.5:0.95, per-class table, confusion matrix.
8. Visualise test-set predictions, provide a reusable `predict_image()` function and save the model.

**Improvements over the reference PDF:** automatic dataset discovery, robust annotation parsing and conversion, `DontCare` handling, a held-out test split, dataset statistics, annotation verification, per-class metrics, custom confusion-matrix plots, error handling at every step, Kaggle-safe paths and modular functions.

> **How to use:** add a KITTI object-detection dataset through *Add Input*, enable a GPU, and *Run All*. Every number shown in this notebook is produced by actually running the code — nothing is hard-coded.

**Two honest caveats about KITTI**
* KITTI frames come from continuous driving sequences, and the public dataset has no sequence IDs in the object-detection split. A random split can therefore place *neighbouring, near-identical frames* in different splits, which makes validation/test scores somewhat optimistic compared with a true sequence-level split.
* `DontCare` regions are not an object class (see Section 4/8), and this simplified pipeline does not mask them during evaluation.

## 2. Environment Setup

Kaggle already ships with PyTorch, OpenCV, pandas, matplotlib, seaborn, etc. This cell installs **only the packages that are missing** (typically just `ultralytics`) and does not touch the large pre-installed packages.

> Installing needs **Internet: On** (Notebook settings → Internet). If everything is already installed, no internet is needed for this cell.

In [ ]:
import importlib
import importlib.util
import subprocess
import sys

# import-name -> pip-name
REQUIRED_PACKAGES = {
    "ultralytics": "ultralytics",
    "cv2": "opencv-python",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "numpy": "numpy",
    "yaml": "PyYAML",
    "PIL": "Pillow",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
}

missing = [pip for mod, pip in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]

if missing:
    print(f"Installing missing packages: {missing}")
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing],
        capture_output=True, text=True,
    )
    if proc.returncode != 0:
        print(proc.stdout[-1500:])
        print(proc.stderr[-1500:])
        raise RuntimeError(
            "Package installation failed. In Kaggle open the right-hand panel "
            "'Session options' -> 'Internet' and switch it ON, then re-run this cell."
        )
    importlib.invalidate_caches()
    print("Installation finished.")
else:
    print("All required packages are already installed - nothing to install.")

## 3. Import Libraries

All imports live in one cell. Weights & Biases logging is disabled (it is not needed and can ask for a login on Kaggle). After importing, the cell prints package versions and checks GPU availability.

In [ ]:
import os, sys, json, math, random, shutil, time, warnings
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Disable Weights & Biases before ultralytics is imported
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import yaml
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import ultralytics
from ultralytics import YOLO, settings as yolo_settings

try:
    yolo_settings.update({"wandb": False})
except Exception:
    pass  # settings keys differ between versions - harmless

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("Package versions")
for name, mod in [("python", None), ("ultralytics", ultralytics), ("torch", torch), ("opencv", cv2),
                  ("numpy", np), ("pandas", pd), ("matplotlib", matplotlib), ("seaborn", sns), ("PyYAML", yaml)]:
    print(f"  {name:12s}: {sys.version.split()[0] if mod is None else mod.__version__}")
print("  Pillow      :", Image.__version__)

# ---- GPU check -------------------------------------------------------
GPU_AVAILABLE = torch.cuda.is_available()
print(f"\nGPU available: {GPU_AVAILABLE}")
if GPU_AVAILABLE:
    print("GPU name:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    DEVICE = 0  # single GPU (avoids multi-GPU/DDP issues inside notebooks)
else:
    DEVICE = "cpu"
    print("WARNING: No GPU detected. Training on CPU will be VERY slow.")
    print("  -> In Kaggle: Session options -> Accelerator -> GPU T4 x2 / P100, then restart the session.")
    print("  -> To still test the notebook on CPU, set MAX_IMAGES = 300 and EPOCHS = 2 in Section 4.")

## 4. Configuration

Everything you may want to tweak is in this single cell.

**Why YOLOv8n first?** The *nano* model has only ~3 M parameters and trains quickly on a free Kaggle GPU, giving a strong, fast baseline. Once the pipeline works you can switch `MODEL_NAME` to `yolov8s.pt` / `yolov8m.pt` for better accuracy at the cost of training time and memory.

**Class handling.** KITTI's 2-D labels contain `Car, Van, Truck, Pedestrian, Person_sitting, Cyclist, Tram, Misc, DontCare`.

| KITTI type | Decision | Reason |
|---|---|---|
| Car, Van, Truck, Pedestrian, Person_sitting, Cyclist, Tram, Misc | **kept** (ids 0–7) | real, detectable road objects |
| DontCare | **ignored** | marks regions that were *not* labelled (far-away or ambiguous objects). It is not an object class; training on it would teach the model that those regions are objects. |
| any other string | **ignored** and reported | prevents silent invalid class ids |

The *single* list `CLASS_NAMES` below is the source of truth for label conversion, `data.yaml`, training, validation, prediction drawing and the confusion matrix.

In [ ]:
# ----------------------------- Reproducibility -----------------------------
SEED = 42

# ----------------------------- Model / training ----------------------------
MODEL_NAME = "yolov8n.pt"      # try "yolov8s.pt" or "yolov8m.pt" for higher accuracy
IMG_SIZE = 640                 # training / inference image size
EPOCHS = 30
BATCH_SIZE = 16                # lower to 8 if you hit a CUDA out-of-memory error
PATIENCE = 10                  # early stopping: epochs without improvement
WORKERS = min(4, os.cpu_count() or 2)
CONF_THRESHOLD = 0.25          # confidence threshold for visualised predictions
EXPERIMENT_NAME = "yolov8n_kitti"

# ----------------------------- Data split ----------------------------------
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.80, 0.10, 0.10
MAX_IMAGES = None              # e.g. 300 for a quick smoke test; None = use all labelled images
MIN_BOX_PX = 1.0               # boxes thinner than this (pixels) are treated as invalid
NUM_PRED_EXAMPLES = 12         # test images shown in the prediction grid

# Optional manual override (normally leave as None - the dataset is auto-detected)
MANUAL_IMAGE_DIR = None
MANUAL_LABEL_DIR = None

# ----------------------------- Paths (Kaggle) ------------------------------
INPUT_ROOT = Path("/kaggle/input")                 # read-only
WORK_ROOT = Path("/kaggle/working")                # writable
DATASET_DIR = WORK_ROOT / "kitti_yolo"             # generated YOLO dataset
STAGING_LABEL_DIR = DATASET_DIR / "_labels_all"    # temporary converted labels
RUNS_DIR = WORK_ROOT / "runs"                      # Ultralytics outputs
OUTPUT_DIR = WORK_ROOT / "final_outputs"           # model + metrics + examples
YAML_PATH = DATASET_DIR / "data.yaml"
DELETE_DATASET_COPY_AT_END = False                 # True frees ~6 GB of output space at the end

# ----------------------------- Classes -------------------------------------
CLASS_NAMES = ["Car", "Pedestrian", "Van", "Cyclist", "Truck", "Misc", "Tram", "Person_sitting"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
LOWER_TO_ID = {name.lower(): i for i, name in enumerate(CLASS_NAMES)}
IGNORED_LOWER = {"dontcare"}
N_CLASSES = len(CLASS_NAMES)

# ----------------------------- Sanity checks -------------------------------
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-6, "TRAIN_FRAC + VAL_FRAC + TEST_FRAC must equal 1.0"
assert IMG_SIZE % 32 == 0, "IMG_SIZE must be a multiple of 32 for YOLO"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
for d in (WORK_ROOT, RUNS_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

EXPECTED_STRUCTURE = """
Expected (flexible) KITTI object-detection layout under /kaggle/input/:

  /kaggle/input/<any-dataset-name>/
      data_object_image_2/training/image_2/000000.png, 000001.png, ...
      data_object_label_2/training/label_2/000000.txt, 000001.txt, ...

Any layout works as long as one folder holds the images and another folder holds
KITTI-format .txt label files whose file names match the image names.
"""

class SetupError(RuntimeError):
    """Raised for configuration / data problems with a human-readable explanation."""

def fail(title, how_to_fix):
    bar = "=" * 80
    print(f"\n{bar}\nERROR: {title}\n{bar}\n{how_to_fix}\n{bar}")
    raise SetupError(title)

print("Classes used for training (id -> name):")
for i, n in enumerate(CLASS_NAMES):
    print(f"  {i}: {n}")
print("Ignored KITTI types:", sorted(IGNORED_LOWER))
print("Model:", MODEL_NAME, "| img size:", IMG_SIZE, "| epochs:", EPOCHS, "| batch:", BATCH_SIZE, "| device:", DEVICE)

## 5. Detect Kaggle Dataset

The notebook walks through `/kaggle/input/`, finds every folder containing images and every folder containing **KITTI-format** label files (≥ 15 space-separated fields per line, which also rejects the `calib` text files), then pairs them by matching file names. The pair with the most matches is selected (`image_2`, the left colour camera, is preferred over `image_3`). Nothing inside `/kaggle/input/` is ever modified.

In [ ]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg"}

def classify_label_dir(label_dir, filenames, n_sample=20):
    """Return 'kitti', 'yolo' or 'other' by inspecting a few label files."""
    kitti_hits, yolo_hits, bad = 0, 0, 0
    names = filenames.values() if isinstance(filenames, dict) else filenames   # dict: stem -> file name
    for fn in sorted(names)[:n_sample]:
        try:
            lines = [l.split() for l in (Path(label_dir) / fn).read_text().splitlines() if l.strip()]
        except Exception:
            bad += 1
            continue
        if not lines:
            continue  # empty file: cannot tell
        first = lines[0]
        try:
            if len(first) >= 15 and not first[0].endswith(":"):
                [float(v) for v in first[1:15]]
                kitti_hits += 1
            elif len(first) in (5, 6) and first[0].isdigit():
                [float(v) for v in first[1:5]]
                yolo_hits += 1
            else:
                bad += 1
        except ValueError:
            bad += 1
    if kitti_hits and not yolo_hits and bad == 0:
        return "kitti"
    if yolo_hits and not kitti_hits and bad == 0:
        return "yolo"
    return "other"


def scan_input_tree(input_root, max_depth=8, min_files=10):
    """Collect folders that contain images and folders that contain .txt files."""
    img_dirs, txt_dirs = {}, {}
    for root, dirs, files in os.walk(input_root):
        depth = len(Path(root).relative_to(input_root).parts)
        if depth >= max_depth:
            dirs[:] = []
        imgs = [f for f in files if Path(f).suffix.lower() in IMAGE_EXTS]
        txts = [f for f in files if f.lower().endswith(".txt")]
        if len(imgs) >= min_files:
            img_dirs[Path(root)] = {Path(f).stem: f for f in imgs}
        if len(txts) >= min_files:
            txt_dirs[Path(root)] = {Path(f).stem: f for f in txts}
    return img_dirs, txt_dirs


def find_kitti_dataset(input_root=INPUT_ROOT, min_matches=10):
    """Auto-detect (image_dir, label_dir). Returns a dict with all discovered info."""
    if not input_root.exists():
        fail("/kaggle/input does not exist",
             "This notebook is designed for Kaggle. Run it inside a Kaggle Notebook.\n" + EXPECTED_STRUCTURE)
    if not any(input_root.iterdir()):
        fail("No dataset is attached to this notebook",
             "Click 'Add Input' (right panel) -> search 'kitti' -> add a KITTI object-detection dataset,\n"
             "e.g. 'klemenko/kitti-dataset' (Left colour images + label_2). Then re-run.\n" + EXPECTED_STRUCTURE)

    # --- manual override -------------------------------------------------------
    if MANUAL_IMAGE_DIR and MANUAL_LABEL_DIR:
        img_dir, lbl_dir = Path(MANUAL_IMAGE_DIR), Path(MANUAL_LABEL_DIR)
        if not img_dir.is_dir() or not lbl_dir.is_dir():
            fail("MANUAL_IMAGE_DIR / MANUAL_LABEL_DIR do not exist", f"{img_dir}\n{lbl_dir}")
        imgs = {p.stem: p.name for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS}
        lbls = {p.stem: p.name for p in lbl_dir.glob("*.txt")}
        matched = sorted(set(imgs) & set(lbls))
        return dict(image_dir=img_dir, label_dir=lbl_dir, image_files=imgs, label_files=lbls,
                    matched=matched, candidates=pd.DataFrame())

    img_dirs, txt_dirs = scan_input_tree(input_root)
    if not img_dirs:
        fail("No image folders found in /kaggle/input",
             "The attached inputs contain no folder with at least 10 .png/.jpg images.\n" + EXPECTED_STRUCTURE)

    kinds = {d: classify_label_dir(d, files) for d, files in txt_dirs.items()}
    candidates = []
    for idir, imgs in img_dirs.items():
        istems = set(imgs)
        for ldir, lbls in txt_dirs.items():
            if kinds[ldir] != "kitti":
                continue
            overlap = len(istems & set(lbls))
            if overlap >= min_matches:
                candidates.append(dict(
                    image_dir=idir, label_dir=ldir, matched_pairs=overlap,
                    n_images=len(imgs), n_labels=len(lbls),
                    is_image_2=int("image_2" in idir.name.lower()),
                    is_training=int("training" in str(idir).lower()),
                ))

    if not candidates:
        yolo_dirs = [d for d, k in kinds.items() if k == "yolo"]
        msg = "Discovered image folders:\n"
        msg += "\n".join(f"  {d}  ({len(f)} images)" for d, f in list(img_dirs.items())[:10])
        msg += "\n\nDiscovered .txt folders (type):\n"
        msg += "\n".join(f"  {d}  ({len(f)} files, {kinds[d]})" for d, f in list(txt_dirs.items())[:10]) or "  (none)"
        if yolo_dirs:
            msg += ("\n\nNote: YOLO-format labels were found, but no KITTI-format labels. This notebook converts "
                    "from the ORIGINAL KITTI labels (label_2). Please add a dataset that contains them.")
        fail("No KITTI image/label pair found", msg + "\n" + EXPECTED_STRUCTURE)

    candidates.sort(key=lambda c: (c["matched_pairs"], c["is_image_2"], c["is_training"]), reverse=True)
    best = candidates[0]
    imgs, lbls = img_dirs[best["image_dir"]], txt_dirs[best["label_dir"]]
    matched = sorted(set(imgs) & set(lbls))
    return dict(image_dir=best["image_dir"], label_dir=best["label_dir"], image_files=imgs, label_files=lbls,
                matched=matched, candidates=pd.DataFrame(candidates).drop(columns=["is_image_2", "is_training"]))


print("Attached inputs:", [p.name for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else "none")
found = find_kitti_dataset()

if len(found["candidates"]) > 1:
    print("\nCandidate image/label pairs detected (best first):")
    display(found["candidates"].head(10))

IMAGE_DIR, LABEL_DIR = found["image_dir"], found["label_dir"]
IMAGE_FILES = found["image_files"]            # stem -> file name
STEMS = list(found["matched"])
print("\nSelected image directory:", IMAGE_DIR)
print("Selected label directory:", LABEL_DIR)

In [ ]:
# ---- Basic counts and consistency checks ----------------------------------------------------
n_images_dir, n_labels_dir = len(found["image_files"]), len(found["label_files"])
images_without_label = sorted(set(found["image_files"]) - set(found["label_files"]))
labels_without_image = sorted(set(found["label_files"]) - set(found["image_files"]))

print(f"Images in folder        : {n_images_dir}")
print(f"Label files in folder   : {n_labels_dir}")
print(f"Matched image/label pairs: {len(STEMS)}")
print(f"Images without a label  : {len(images_without_label)}  (excluded)")
print(f"Labels without an image : {len(labels_without_image)}  (excluded)")

if len(STEMS) == 0:
    fail("Empty dataset - no image has a matching label file", EXPECTED_STRUCTURE)
if len(STEMS) < 20:
    fail("Too few labelled images to split into train/val/test", f"Only {len(STEMS)} labelled images found.")
if images_without_label:
    print("  e.g. unlabeled images:", images_without_label[:5])

if MAX_IMAGES is not None and MAX_IMAGES < len(STEMS):
    STEMS = sorted(random.Random(SEED).sample(STEMS, MAX_IMAGES))
    print(f"\nMAX_IMAGES set -> using a reproducible random subset of {len(STEMS)} images.")

DATASET_ROOT = INPUT_ROOT / IMAGE_DIR.relative_to(INPUT_ROOT).parts[0]
print("\nDataset root:", DATASET_ROOT)

## 6. Explore Dataset Structure

First a compact directory tree of the attached dataset (file counts per extension instead of thousands of names), followed by a few raw sample images.

In [ ]:
def print_tree(root, max_depth=3, max_dirs=12):
    """Print a compact directory tree: sub-folders plus a file-count summary per folder."""
    root = Path(root)

    def walk(d, prefix, depth):
        try:
            entries = sorted(d.iterdir(), key=lambda p: (p.is_file(), p.name))
        except PermissionError:
            return
        dirs = [e for e in entries if e.is_dir()]
        files = [e for e in entries if e.is_file()]
        if files:
            ext = Counter(f.suffix.lower() or "(no ext)" for f in files)
            print(f"{prefix}[{len(files)} files: " + ", ".join(f"{n} x {e}" for e, n in ext.most_common(4)) + "]")
        for sub in dirs[:max_dirs]:
            print(f"{prefix}{sub.name}/")
            if depth < max_depth:
                walk(sub, prefix + "    ", depth + 1)
        if len(dirs) > max_dirs:
            print(f"{prefix}... (+{len(dirs) - max_dirs} more folders)")

    print(f"{root.name}/")
    walk(root, "    ", 1)

print_tree(DATASET_ROOT, max_depth=4)

In [ ]:
def load_image_rgb(path):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f"Could not read image (missing or corrupted): {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def show_grid(images, titles, ncols=2, size_per_image=(8, 2.8), suptitle=None, save_path=None):
    """Display a list of RGB images in a clean grid."""
    n = len(images)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(size_per_image[0] * ncols, size_per_image[1] * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
    if suptitle:
        fig.suptitle(suptitle, fontsize=14)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=110, bbox_inches="tight")
    plt.show()


sample_stems = random.Random(SEED).sample(STEMS, 4)
imgs = [load_image_rgb(IMAGE_DIR / IMAGE_FILES[s]) for s in sample_stems]
show_grid(imgs, [f"{IMAGE_FILES[s]}  -  {im.shape[1]}x{im.shape[0]} px" for s, im in zip(sample_stems, imgs)],
          suptitle="Raw sample images")

## 7. Dataset Analysis

Every image is opened once to read its true width/height and to detect **corrupted files** (they are excluded and listed). Then every KITTI label file is parsed into a table (`df_obj`), one row per annotated object.

In [ ]:
def inspect_image(path):
    """Return (width, height, ok). Detects corrupted / unreadable images."""
    try:
        with Image.open(path) as im:
            w, h = im.size
            im.verify()
        return w, h, True
    except Exception:
        return 0, 0, False


paths = [IMAGE_DIR / IMAGE_FILES[s] for s in STEMS]
with ThreadPoolExecutor(8) as ex:
    res = list(tqdm(ex.map(inspect_image, paths), total=len(paths), desc="Checking images"))

df_img = pd.DataFrame({"stem": STEMS, "width": [r[0] for r in res], "height": [r[1] for r in res],
                       "ok": [r[2] for r in res]})
corrupt = df_img[~df_img["ok"]]
if len(corrupt):
    print(f"WARNING: {len(corrupt)} corrupted/unreadable images were excluded, e.g. {corrupt['stem'].head(5).tolist()}")
df_img = df_img[df_img["ok"]].reset_index(drop=True)
STEMS = df_img["stem"].tolist()
if not STEMS:
    fail("All images are unreadable", "Re-attach the dataset or check that the images are valid PNG/JPEG files.")
IMG_SIZES = {r.stem: (int(r.width), int(r.height)) for r in df_img.itertuples()}
print(f"Valid images: {len(STEMS)}")

In [ ]:
KITTI_FIELDS = ["type", "truncated", "occluded", "alpha", "bbox_left", "bbox_top", "bbox_right", "bbox_bottom",
                "dim_height", "dim_width", "dim_length", "loc_x", "loc_y", "loc_z", "rotation_y"]

def parse_kitti_annotation(label_path):
    """Parse one KITTI label file. Returns (objects, n_malformed_lines).

    Each object keeps the 2-D information needed for detection:
    type, truncated, occluded, alpha and the pixel box (x1, y1, x2, y2).
    """
    objects, bad = [], 0
    try:
        text = Path(label_path).read_text()
    except Exception:
        return objects, 1
    for line in text.splitlines():
        parts = line.split()
        if not parts:
            continue
        if len(parts) < 8:
            bad += 1
            continue
        try:
            objects.append(dict(
                type=parts[0], truncated=float(parts[1]), occluded=int(float(parts[2])), alpha=float(parts[3]),
                x1=float(parts[4]), y1=float(parts[5]), x2=float(parts[6]), y2=float(parts[7]),
            ))
        except ValueError:
            bad += 1
    return objects, bad


def status_of(kitti_type):
    t = kitti_type.lower()
    if t in LOWER_TO_ID:
        return "kept"
    if t in IGNORED_LOWER:
        return "ignored (DontCare)"
    return "ignored (unknown type)"


ANNOTATIONS, bad_lines = {}, 0
for s in tqdm(STEMS, desc="Parsing KITTI labels"):
    objs, bad = parse_kitti_annotation(LABEL_DIR / f"{s}.txt")
    ANNOTATIONS[s] = objs
    bad_lines += bad

rows = [dict(stem=s, **o) for s, objs in ANNOTATIONS.items() for o in objs]
if not rows:
    fail("The label files contain no annotated objects", "Check that the label folder holds real KITTI label_2 files.")
df_obj = pd.DataFrame(rows)
df_obj["status"] = df_obj["type"].map(status_of)
df_obj["box_w"] = df_obj["x2"] - df_obj["x1"]
df_obj["box_h"] = df_obj["y2"] - df_obj["y1"]
if bad_lines:
    print(f"WARNING: {bad_lines} malformed label lines were skipped.")
print(f"Parsed {len(df_obj)} objects from {len(ANNOTATIONS)} label files.")
df_obj.head()

In [ ]:
kept = df_obj[df_obj["status"] == "kept"]
kept_per_image = kept.groupby("stem").size().reindex(STEMS, fill_value=0)

summary = {
    "Number of images": len(STEMS),
    "Number of annotation files": len(ANNOTATIONS),
    "Total annotated objects (raw)": len(df_obj),
    "Objects kept for training": len(kept),
    "Objects ignored (DontCare / unknown)": int((df_obj["status"] != "kept").sum()),
    "Images with no kept object": int((kept_per_image == 0).sum()),
    "Objects per image - mean / median / max": f"{kept_per_image.mean():.2f} / {kept_per_image.median():.0f} / {kept_per_image.max()}",
}
print(pd.Series(summary).to_string())

size_table = (df_img.groupby(["width", "height"]).size().reset_index(name="n_images")
              .sort_values("n_images", ascending=False).reset_index(drop=True))
print("\nImage dimensions (width x height):")
display(size_table.head(10))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
bins = np.arange(0, kept_per_image.max() + 2) - 0.5
axes[0].hist(kept_per_image, bins=bins, color="#4C72B0", edgecolor="white")
axes[0].set(title="Kept objects per image", xlabel="objects in image", ylabel="number of images")

top = size_table.head(6).copy()
top["size"] = top["width"].astype(str) + "x" + top["height"].astype(str)
sns.barplot(data=top, x="size", y="n_images", ax=axes[1], color="#55A868")
axes[1].set(title="Most common image sizes", xlabel="width x height", ylabel="number of images")
for p in axes[1].patches:
    axes[1].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 8. KITTI Annotation Analysis

A KITTI label line has 15 values: `type truncated occluded alpha bbox(4) dimensions(3) location(3) rotation_y`. Object detection only needs the **type** and the **2-D box** (`bbox_left, bbox_top, bbox_right, bbox_bottom` in pixels).

In [ ]:
first_stem = STEMS[0]
print(f"Raw content of {first_stem}.txt:\n")
print((LABEL_DIR / f"{first_stem}.txt").read_text())

field_info = pd.DataFrame({
    "field": KITTI_FIELDS,
    "meaning": ["object class", "0 (fully visible) .. 1 (leaving the image)", "0 visible, 1 partly, 2 largely, 3 unknown",
                "observation angle (rad)", "box left (px)", "box top (px)", "box right (px)", "box bottom (px)",
                "3-D height (m)", "3-D width (m)", "3-D length (m)", "3-D x (m)", "3-D y (m)", "3-D z (m)",
                "yaw (rad)"],
    "used here": ["yes", "no", "no", "no", "yes", "yes", "yes", "yes"] + ["no"] * 7,
})
display(field_info)

print(f"\nSample objects of {first_stem} (parsed):")
display(df_obj[df_obj["stem"] == first_stem][["type", "truncated", "occluded", "x1", "y1", "x2", "y2", "status"]])

In [ ]:
type_counts = df_obj["type"].value_counts()
class_table = pd.DataFrame({"objects": type_counts})
class_table["share_%"] = (100 * class_table["objects"] / class_table["objects"].sum()).round(2)
class_table["decision"] = [status_of(t) for t in class_table.index]
class_table["yolo_id"] = [LOWER_TO_ID.get(t.lower(), "-") for t in class_table.index]
print("Object classes found in the dataset:")
display(class_table)

unknown = class_table[class_table["decision"] == "ignored (unknown type)"]
if len(unknown):
    print("WARNING: unknown types found and ignored:", unknown.index.tolist())
missing_classes = [c for c in CLASS_NAMES if c.lower() not in {t.lower() for t in type_counts.index}]
if missing_classes:
    print("NOTE: classes in CLASS_NAMES that do not occur in this dataset:", missing_classes)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
palette = {"kept": "#4C72B0", "ignored (DontCare)": "#C44E52", "ignored (unknown type)": "#8172B2"}
plot_df = class_table.reset_index().rename(columns={"index": "type"})
sns.barplot(data=plot_df, y="type", x="objects", hue="decision", dodge=False, palette=palette, ax=axes[0])
axes[0].set(title="Class distribution (raw KITTI types)", xlabel="number of objects", ylabel="")
for p in axes[0].patches:
    w = p.get_width()
    if w > 0:
        axes[0].annotate(f"{int(w)}", (w, p.get_y() + p.get_height() / 2), va="center", ha="left", fontsize=9)
axes[0].set_xlim(0, plot_df["objects"].max() * 1.12)

axes[1].hist(kept["box_h"].clip(upper=400), bins=40, color="#55A868", edgecolor="white")
axes[1].set(title="Bounding-box height of kept objects (clipped at 400 px)", xlabel="box height (px)",
            ylabel="number of objects")
axes[1].axvline(kept["box_h"].median(), color="k", ls="--", label=f"median = {kept['box_h'].median():.0f} px")
axes[1].legend()
plt.tight_layout()
plt.show()
print(f"{(kept['box_h'] < 25).mean() * 100:.1f}% of kept boxes are shorter than 25 px in the original image "
      f"-> small objects are hard, which is why a larger IMG_SIZE can help.")

In [ ]:
PALETTE = [tuple(int(c * 255) for c in plt.cm.tab10(i)[:3]) for i in range(10)]

def draw_boxes(image_rgb, items, thickness=None, font_scale=None):
    """Draw boxes on a copy of the image.

    items: iterable of (x1, y1, x2, y2, text, color_index)
    """
    img = image_rgb.copy()
    h, w = img.shape[:2]
    thickness = thickness or max(2, int(round(w / 600)))
    font_scale = font_scale or max(0.6, w / 1242 * 0.85)
    text_th = max(1, thickness - 1)
    for x1, y1, x2, y2, text, ci in items:
        color = PALETTE[int(ci) % len(PALETTE)]
        p1, p2 = (int(round(x1)), int(round(y1))), (int(round(x2)), int(round(y2)))
        cv2.rectangle(img, p1, p2, color, thickness)
        if text:
            (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_th)
            ty = p1[1] - 4 if p1[1] - th - base - 4 >= 0 else p1[1] + th + 4
            tx = min(p1[0], max(0, w - tw - 2))
            cv2.rectangle(img, (tx, ty - th - base), (tx + tw + 2, ty + base), color, -1)
            cv2.putText(img, text, (tx + 1, ty), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), text_th,
                        cv2.LINE_AA)
    return img


def visualize_annotations(stems, n=6, show_coords=False, save_path=None, suptitle="Ground-truth annotations (kept classes)"):
    """Draw the KITTI ground-truth boxes (kept classes only) for several images."""
    chosen = list(stems)[:n]
    out_imgs, titles = [], []
    for s in chosen:
        img = load_image_rgb(IMAGE_DIR / IMAGE_FILES[s])
        items = []
        for o in ANNOTATIONS[s]:
            cid = LOWER_TO_ID.get(o["type"].lower())
            if cid is None:
                continue
            txt = CLASS_NAMES[cid]
            if show_coords:
                txt += f" ({o['x1']:.0f},{o['y1']:.0f},{o['x2']:.0f},{o['y2']:.0f})"
            items.append((o["x1"], o["y1"], o["x2"], o["y2"], txt, cid))
        out_imgs.append(draw_boxes(img, items))
        titles.append(f"{IMAGE_FILES[s]} - {len(items)} objects")
    show_grid(out_imgs, titles, suptitle=suptitle, save_path=save_path)


vis_stems = random.Random(SEED + 1).sample(STEMS, 6)
visualize_annotations(vis_stems, n=6)

## 9. KITTI to YOLO Conversion

For every image the pixel box `(x1, y1, x2, y2)` becomes

```
x_center = ((x1 + x2) / 2) / image_width
y_center = ((y1 + y2) / 2) / image_height
width    = (x2 - x1) / image_width
height   = (y2 - y1) / image_height
```

and the KITTI type is mapped to its class id via `CLASS_NAMES`. Safety rules applied:

* `DontCare` and unknown types are **not** written to the label files.
* Boxes are **clipped** to the image; boxes with zero/negative size after clipping are dropped and counted.
* Images with no kept object get an **empty** label file (valid YOLO "background" image).

Labels are first written to a temporary staging folder inside `/kaggle/working/` and moved into the split folders in Section 11.

In [ ]:
def kitti_box_to_yolo(x1, y1, x2, y2, img_w, img_h, min_px=MIN_BOX_PX):
    """Convert a pixel box to normalised YOLO (xc, yc, w, h). Returns None for invalid boxes."""
    x1, x2 = sorted((x1, x2))
    y1, y2 = sorted((y1, y2))
    x1, x2 = max(0.0, min(x1, img_w)), max(0.0, min(x2, img_w))   # clip to image
    y1, y2 = max(0.0, min(y1, img_h)), max(0.0, min(y2, img_h))
    bw, bh = x2 - x1, y2 - y1
    if bw < min_px or bh < min_px:
        return None
    return ((x1 + x2) / 2 / img_w, (y1 + y2) / 2 / img_h, bw / img_w, bh / img_h)


def convert_kitti_to_yolo(stems, annotations, sizes, out_dir):
    """Write one YOLO label file per image into out_dir. Returns a Counter with conversion statistics."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stats = Counter()
    for stem in tqdm(stems, desc="Converting labels"):
        w, h = sizes[stem]
        lines = []
        for o in annotations[stem]:
            key = o["type"].lower()
            cid = LOWER_TO_ID.get(key)
            if cid is None:
                stats["ignored_dontcare" if key in IGNORED_LOWER else "ignored_unknown_type"] += 1
                continue
            if o["x1"] < 0 or o["y1"] < 0 or o["x2"] > w or o["y2"] > h:
                stats["boxes_clipped"] += 1
            box = kitti_box_to_yolo(o["x1"], o["y1"], o["x2"], o["y2"], w, h)
            if box is None:
                stats["dropped_invalid_box"] += 1
                continue
            lines.append(f"{cid} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}")
            stats["converted"] += 1
            stats[f"class::{CLASS_NAMES[cid]}"] += 1
        (out_dir / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
        stats["empty_label_files"] += int(not lines)
    return stats


# Start from a clean generated dataset folder (only ever inside /kaggle/working)
assert str(DATASET_DIR).startswith("/kaggle/working"), "Refusing to delete outside /kaggle/working"
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)

conv_stats = convert_kitti_to_yolo(STEMS, ANNOTATIONS, IMG_SIZES, STAGING_LABEL_DIR)

print("\nConversion statistics")
for k in ["converted", "ignored_dontcare", "ignored_unknown_type", "dropped_invalid_box", "boxes_clipped", "empty_label_files"]:
    print(f"  {k:24s}: {conv_stats.get(k, 0)}")
print("\nConverted objects per class:")
for name in CLASS_NAMES:
    print(f"  {name:15s}: {conv_stats.get('class::' + name, 0)}")
if conv_stats.get("dropped_invalid_box"):
    print("\nNOTE: some boxes were degenerate (zero width/height after clipping) and were dropped.")

## 10. Verify Converted Annotations

Three independent checks:

1. **Format check** - every line has 5 values, class id in `[0, N_CLASSES)`, all coordinates in `[0, 1]`, positive width/height.
2. **Numerical round-trip** - converting the YOLO boxes back to pixels must reproduce the (clipped) KITTI boxes (error should be ≈ 0).
3. **Visual check** - converted boxes are drawn back onto the images.

In [ ]:
def read_yolo_label(path):
    """Read a YOLO label file into an (N, 5) float array [class, xc, yc, w, h]."""
    rows = []
    for line in Path(path).read_text().splitlines():
        p = line.split()
        if p:
            rows.append([float(v) for v in p[:5]])
    return np.array(rows, dtype=float).reshape(-1, 5)


def check_yolo_labels(label_dir, n_classes):
    """Validate all YOLO label files in a folder. Returns dict(n_files, n_boxes, issues)."""
    issues, n_files, n_boxes = [], 0, 0
    for f in sorted(Path(label_dir).glob("*.txt")):
        n_files += 1
        for ln, line in enumerate(f.read_text().splitlines(), 1):
            if not line.strip():
                continue
            p = line.split()
            try:
                if len(p) != 5:
                    raise ValueError(f"expected 5 values, got {len(p)}")
                cid, vals = int(p[0]), [float(v) for v in p[1:]]
            except ValueError as e:
                issues.append((f.name, ln, f"malformed line: {e}"))
                continue
            if not 0 <= cid < n_classes:
                issues.append((f.name, ln, f"invalid class id {cid}"))
            if any(v < 0 or v > 1 for v in vals):
                issues.append((f.name, ln, "coordinate outside [0, 1]"))
            if vals[2] <= 0 or vals[3] <= 0:
                issues.append((f.name, ln, "non-positive width/height"))
            n_boxes += 1
    return dict(n_files=n_files, n_boxes=n_boxes, issues=issues)


# 1) format check
chk = check_yolo_labels(STAGING_LABEL_DIR, N_CLASSES)
print(f"Checked {chk['n_files']} label files / {chk['n_boxes']} boxes -> {len(chk['issues'])} issues")
if chk["issues"]:
    print("First issues:", chk["issues"][:5])
    fail("Converted labels contain invalid entries", "See the issues above; check the raw KITTI labels for those files.")

# 2) numerical round trip on a sample
errs, mismatches = [], 0
for s in random.Random(SEED).sample(STEMS, min(300, len(STEMS))):
    w, h = IMG_SIZES[s]
    expected = []
    for o in ANNOTATIONS[s]:
        cid = LOWER_TO_ID.get(o["type"].lower())
        if cid is not None and kitti_box_to_yolo(o["x1"], o["y1"], o["x2"], o["y2"], w, h) is not None:
            x1, x2 = sorted((max(0, min(o["x1"], w)), max(0, min(o["x2"], w))))
            y1, y2 = sorted((max(0, min(o["y1"], h)), max(0, min(o["y2"], h))))
            expected.append((cid, x1, y1, x2, y2))
    lab = read_yolo_label(STAGING_LABEL_DIR / f"{s}.txt")
    if len(lab) != len(expected):
        mismatches += 1
        continue
    for row, (cid, x1, y1, x2, y2) in zip(lab, expected):
        xc, yc, bw, bh = row[1] * w, row[2] * h, row[3] * w, row[4] * h
        errs.append(max(abs(xc - bw / 2 - x1), abs(yc - bh / 2 - y1), abs(xc + bw / 2 - x2), abs(yc + bh / 2 - y2)))
        assert int(row[0]) == cid, f"class id mismatch in {s}"
print(f"Round-trip: {len(errs)} boxes compared, max pixel error = {max(errs) if errs else float('nan'):.5f} px, "
      f"images with box-count mismatch = {mismatches}")
assert mismatches == 0 and (not errs or max(errs) < 0.01), "Conversion verification failed!"
print("Conversion verified OK.")

In [ ]:
def draw_yolo_labels(image_path, label_path):
    """Draw YOLO-format labels (normalised) back onto an image."""
    img = load_image_rgb(image_path)
    h, w = img.shape[:2]
    items = []
    for cid, xc, yc, bw, bh in read_yolo_label(label_path):
        items.append(((xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h,
                      CLASS_NAMES[int(cid)], int(cid)))
    return draw_boxes(img, items), len(items)


imgs, titles = [], []
for s in random.Random(SEED + 1).sample(STEMS, 4):          # same seed as Section 8 -> compare directly
    im, n = draw_yolo_labels(IMAGE_DIR / IMAGE_FILES[s], STAGING_LABEL_DIR / f"{s}.txt")
    imgs.append(im); titles.append(f"{IMAGE_FILES[s]} - {n} converted YOLO boxes")
show_grid(imgs, titles, suptitle="Converted YOLO labels drawn back onto the images")

## 11. Train / Validation / Test Split

* Fixed `SEED` -> the split is identical on every run.
* Default **80 % train / 10 % validation / 10 % test**.
* Each image is assigned to exactly **one** split; the check at the end proves the three sets are disjoint.
* Images are *copied* (never moved) from `/kaggle/input/`, which is read-only; labels are moved from the staging folder, so every image has its matching label file.

Resulting structure:

```
/kaggle/working/kitti_yolo/
    images/{train,val,test}/
    labels/{train,val,test}/
    data.yaml
```

> Reminder: the split is random at frame level; consecutive KITTI frames can be very similar (see the caveat in Section 1).

In [ ]:
def split_dataset(stems, train_frac, val_frac, seed):
    """Reproducible random split of image stems into train / val / test."""
    s = sorted(stems)
    random.Random(seed).shuffle(s)
    n = len(s)
    n_train = int(round(n * train_frac))
    n_val = int(round(n * val_frac))
    splits = {"train": s[:n_train], "val": s[n_train:n_train + n_val], "test": s[n_train + n_val:]}
    if any(len(v) == 0 for v in splits.values()):
        fail("A split ended up empty", "Use more images or adjust TRAIN_FRAC / VAL_FRAC / TEST_FRAC.")
    return splits


def _copy_one(args):
    src, dst = args
    shutil.copy2(src, dst)


def materialize_splits(splits):
    """Create the YOLO folder structure, copy images, move labels."""
    for split, stems in splits.items():
        (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
        jobs = [(IMAGE_DIR / IMAGE_FILES[s], DATASET_DIR / "images" / split / IMAGE_FILES[s]) for s in stems]
        with ThreadPoolExecutor(8) as ex:
            list(tqdm(ex.map(_copy_one, jobs), total=len(jobs), desc=f"Copying {split} images"))
        for s in stems:
            shutil.move(str(STAGING_LABEL_DIR / f"{s}.txt"), str(DATASET_DIR / "labels" / split / f"{s}.txt"))


SPLITS = split_dataset(STEMS, TRAIN_FRAC, VAL_FRAC, SEED)
materialize_splits(SPLITS)
if not any(STAGING_LABEL_DIR.iterdir()):
    STAGING_LABEL_DIR.rmdir()

# ---- verification -----------------------------------------------------------------------------
all_sets = {k: set(v) for k, v in SPLITS.items()}
assert not (all_sets["train"] & all_sets["val"]) and not (all_sets["train"] & all_sets["test"]) \
    and not (all_sets["val"] & all_sets["test"]), "Data leakage: a stem appears in several splits!"

for split in SPLITS:
    img_files = sorted((DATASET_DIR / "images" / split).iterdir())
    lbl_files = {p.stem for p in (DATASET_DIR / "labels" / split).glob("*.txt")}
    no_label = [p.name for p in img_files if p.stem not in lbl_files]
    if no_label:
        fail(f"Images without labels in split '{split}'", f"Examples: {no_label[:5]}")

print(f"Train images:      {len(SPLITS['train'])}")
print(f"Validation images: {len(SPLITS['val'])}")
print(f"Test images:       {len(SPLITS['test'])}")
print("Every image has a matching label file, and the splits are disjoint.")

## 12. Create data.yaml

The YAML is generated from Python variables (no hard-coded paths) using the same `CLASS_NAMES` list used everywhere else. It is re-loaded and validated before being printed.

In [ ]:
def create_yaml(dataset_dir, class_names, yaml_path):
    cfg = {
        "path": str(dataset_dir),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {i: n for i, n in enumerate(class_names)},
    }
    with open(yaml_path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return cfg


def validate_yaml(yaml_path):
    if not Path(yaml_path).exists():
        fail("data.yaml is missing", "Re-run Section 12 to regenerate it.")
    cfg = yaml.safe_load(Path(yaml_path).read_text())
    for key in ("train", "val", "test"):
        d = Path(cfg["path"]) / cfg[key]
        if not d.is_dir():
            fail(f"data.yaml '{key}' folder does not exist", str(d))
    if [cfg["names"][i] for i in range(len(cfg["names"]))] != CLASS_NAMES:
        fail("Class names in data.yaml differ from CLASS_NAMES", "Re-run Section 12.")
    return cfg


create_yaml(DATASET_DIR, CLASS_NAMES, YAML_PATH)
DATA_CFG = validate_yaml(YAML_PATH)
print(f"data.yaml saved to: {YAML_PATH}\n")
print(YAML_PATH.read_text())

## 13. Visualize YOLO Dataset

A final look at the dataset that will actually be used for training: instances per class in each split (a random split is not stratified, so rare classes such as `Person_sitting` have few examples in the smaller splits), plus a few training images with their YOLO labels.

In [ ]:
def count_instances(split):
    """Count instances per class name in a split's YOLO label files."""
    counts = Counter()
    for f in (DATASET_DIR / "labels" / split).glob("*.txt"):
        for row in read_yolo_label(f):
            counts[CLASS_NAMES[int(row[0])]] += 1
    return counts


split_counts = pd.DataFrame({sp: pd.Series(count_instances(sp)) for sp in ["train", "val", "test"]}) \
                 .reindex(CLASS_NAMES).fillna(0).astype(int)
split_counts.loc["TOTAL"] = split_counts.sum()
print("Instances per class and split:")
display(split_counts)

share = split_counts.drop(index="TOTAL")
share = share / share.sum()
ax = share.plot(kind="bar", figsize=(12, 4.5), width=0.8)
ax.set(title="Class share within each split (distributions should look similar)", ylabel="fraction of instances")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

vis = random.Random(SEED + 2).sample(sorted((DATASET_DIR / "images" / "train").iterdir()), 4)
imgs, titles = [], []
for p in vis:
    im, n = draw_yolo_labels(p, DATASET_DIR / "labels" / "train" / f"{p.stem}.txt")
    imgs.append(im); titles.append(f"train/{p.name} - {n} boxes")
show_grid(imgs, titles, suptitle="Training images with YOLO labels")

## 14. Load YOLOv8 Model

The pretrained COCO weights (`MODEL_NAME`) are loaded; Ultralytics automatically replaces the 80-class detection head with an 8-class head when training with our `data.yaml`.

Weights are normally downloaded from GitHub (needs **Internet: On**). With internet off, the loader first looks for a file with that name in `/kaggle/working/` and in all attached inputs (e.g. a dataset that contains `yolov8n.pt`) and otherwise explains how to fix it.

In [ ]:
def load_yolo_model(weights_name):
    """Load YOLO weights from a local/attached file if present, otherwise let Ultralytics download them."""
    p = Path(weights_name)
    for c in (p, Path.cwd() / p.name, WORK_ROOT / p.name):
        if c.is_file():
            print("Using local weights:", c)
            return YOLO(str(c))
    for root, _, files in os.walk(INPUT_ROOT):
        if p.name in files:
            print("Using weights found in attached input:", Path(root) / p.name)
            return YOLO(str(Path(root) / p.name))
    try:
        print(f"Downloading {p.name} (needs internet) ...")
        return YOLO(weights_name)
    except Exception as e:
        fail(f"Model weights '{weights_name}' are not available",
             f"Original error: {e}\n\nFix (choose one):\n"
             "  1) Turn Internet ON in the Kaggle notebook settings and re-run, or\n"
             f"  2) Add a Kaggle dataset containing '{p.name}' via 'Add Input' and re-run.")


model = load_yolo_model(MODEL_NAME)
n_params = sum(p.numel() for p in model.model.parameters())
print(f"\nLoaded {MODEL_NAME}: {n_params / 1e6:.2f} M parameters (COCO-pretrained, 80 classes; head is replaced for training)")

## 15. Train Model

Fine-tuning settings:

| Setting | Value | Why |
|---|---|---|
| pretrained weights | COCO `yolov8n.pt` | transfer learning converges much faster than training from scratch |
| `imgsz` | `IMG_SIZE` | KITTI images are very wide (~1242x375); they are letter-boxed to this size |
| `epochs` / `patience` | `EPOCHS` / `PATIENCE` | early stopping if validation fitness stops improving |
| `batch` | `BATCH_SIZE` | 16 fits a 16 GB GPU at 640 px for the nano model |
| `seed` | `SEED` | reproducibility |
| output | `/kaggle/working/runs/<EXPERIMENT_NAME>` | persisted in the notebook output |

Training prints one progress block per epoch. On a Kaggle GPU this typically takes tens of minutes for 30 epochs; on CPU it would take many hours.

In [ ]:
def train_model(model, data_yaml, epochs, imgsz, batch, patience, device, workers, project, name, seed):
    """Train and return (train_dir, best_pt_path)."""
    if not Path(data_yaml).exists():
        fail("data.yaml not found", "Run Sections 11-12 first.")
    try:
        model.train(
            data=str(data_yaml), epochs=epochs, imgsz=imgsz, batch=batch, patience=patience,
            device=device, workers=workers, project=str(project), name=name, exist_ok=True,
            seed=seed, pretrained=True, plots=True, verbose=True,
        )
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("\nCUDA out of memory. Fix: lower BATCH_SIZE (e.g. 8) or IMG_SIZE (e.g. 512) in Section 4, "
                  "restart the session and re-run.")
        raise
    train_dir = Path(model.trainer.save_dir)
    best = train_dir / "weights" / "best.pt"
    if not best.exists():                                     # defensive fallback
        found_best = sorted(Path(project).rglob("best.pt"), key=lambda p: p.stat().st_mtime)
        if not found_best:
            fail("best.pt was not produced", f"Look inside {project} for the training log.")
        best = found_best[-1]
        train_dir = best.parent.parent
    return train_dir, best


t0 = time.time()
TRAIN_DIR, BEST_PT = train_model(model, YAML_PATH, EPOCHS, IMG_SIZE, BATCH_SIZE, PATIENCE, DEVICE, WORKERS,
                                 RUNS_DIR, EXPERIMENT_NAME, SEED)
print(f"\nTraining finished in {(time.time() - t0) / 60:.1f} min")
print("Run directory:", TRAIN_DIR)
print("Best weights :", BEST_PT)

## 16. Training Results

Ultralytics logs one row per epoch in `results.csv`. The curves below are drawn from that file.

**Metrics explained**

* **box loss** - how well predicted box coordinates match the ground truth (CIoU-based). Lower is better.
* **cls loss** - classification error of the predicted class. Lower is better.
* **DFL loss** - *Distribution Focal Loss*, refines box-edge localisation. Lower is better.
* **Precision** - of all detections the model makes, the fraction that are correct (few false alarms).
* **Recall** - of all real objects, the fraction the model finds (few misses).
* **mAP@0.5** - mean (over classes) average precision when a detection counts as correct if IoU ≥ 0.5.
* **mAP@0.5:0.95** - the same, averaged over IoU thresholds 0.5, 0.55, ..., 0.95; a stricter measure of localisation quality and the headline YOLO metric.

A widening gap between falling training loss and flat/rising validation loss indicates over-fitting.

In [ ]:
res_csv = TRAIN_DIR / "results.csv"
if not res_csv.exists():
    fail("results.csv not found", f"Expected at {res_csv}. Training may not have completed.")
df_res = pd.read_csv(res_csv)
df_res.columns = [c.strip() for c in df_res.columns]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for col, lab in [("train/box_loss", "box"), ("train/cls_loss", "cls"), ("train/dfl_loss", "dfl")]:
    if col in df_res: axes[0, 0].plot(df_res["epoch"], df_res[col], marker="o", ms=3, label=lab)
axes[0, 0].set(title="Training losses", xlabel="epoch"); axes[0, 0].legend()
for col, lab in [("val/box_loss", "box"), ("val/cls_loss", "cls"), ("val/dfl_loss", "dfl")]:
    if col in df_res: axes[0, 1].plot(df_res["epoch"], df_res[col], marker="o", ms=3, label=lab)
axes[0, 1].set(title="Validation losses", xlabel="epoch"); axes[0, 1].legend()
for col, lab in [("metrics/precision(B)", "precision"), ("metrics/recall(B)", "recall")]:
    if col in df_res: axes[1, 0].plot(df_res["epoch"], df_res[col], marker="o", ms=3, label=lab)
axes[1, 0].set(title="Precision / Recall (validation)", xlabel="epoch", ylim=(0, 1)); axes[1, 0].legend()
for col, lab in [("metrics/mAP50(B)", "mAP@0.5"), ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]:
    if col in df_res: axes[1, 1].plot(df_res["epoch"], df_res[col], marker="o", ms=3, label=lab)
axes[1, 1].set(title="mAP (validation)", xlabel="epoch", ylim=(0, 1)); axes[1, 1].legend()
plt.tight_layout(); plt.show()

fit = 0.1 * df_res["metrics/mAP50(B)"] + 0.9 * df_res["metrics/mAP50-95(B)"]
best_epoch = int(df_res.loc[fit.idxmax(), "epoch"])
print(f"Epochs completed: {len(df_res)} (configured: {EPOCHS}) | best epoch by fitness: {best_epoch}")
display(df_res.tail(3).round(4))

rp = TRAIN_DIR / "results.png"
if rp.exists():
    plt.figure(figsize=(14, 7)); plt.imshow(Image.open(rp)); plt.axis("off")
    plt.title("Ultralytics results.png"); plt.show()

## 17. Validate Model

Validation always uses **`best.pt`** (the checkpoint with the best validation fitness), loaded into a fresh model object. We evaluate on:

* the **validation** split (also used for checkpoint selection, so slightly optimistic), and
* the **test** split, which was never used for training or model selection - the fairer estimate.

We also assert that the class names stored in the weights equal `CLASS_NAMES`.

In [ ]:
def evaluate_model(model, split, run_name):
    """Run Ultralytics validation on a split. Returns (results, headline_metrics_dict, per_class_df)."""
    res = model.val(data=str(YAML_PATH), split=split, imgsz=IMG_SIZE, batch=BATCH_SIZE, device=DEVICE,
                    workers=WORKERS, project=str(RUNS_DIR), name=run_name, exist_ok=True, plots=True, verbose=False)
    metrics = {
        "Precision": float(res.box.mp),
        "Recall": float(res.box.mr),
        "mAP50": float(res.box.map50),
        "mAP50-95": float(res.box.map),
    }
    inst = count_instances(split)
    rows = {c: dict(Class=c, Instances=int(inst.get(c, 0)), Precision=np.nan, Recall=np.nan,
                    mAP50=np.nan, **{"mAP50-95": np.nan}) for c in CLASS_NAMES}
    for k, cid in enumerate(res.box.ap_class_index):
        p, r, ap50, ap = res.box.class_result(k)
        rows[CLASS_NAMES[int(cid)]].update({"Precision": float(p), "Recall": float(r),
                                            "mAP50": float(ap50), "mAP50-95": float(ap)})
    return res, metrics, pd.DataFrame(rows.values())


if not BEST_PT.exists():
    fail("best.pt not found", f"Expected at {BEST_PT}. Re-run Section 15.")
best_model = YOLO(str(BEST_PT))
assert [best_model.names[i] for i in range(len(best_model.names))] == CLASS_NAMES, "Class names in weights differ from CLASS_NAMES"

print("=== Validation split ===")
val_res, VAL_METRICS, VAL_PER_CLASS = evaluate_model(best_model, "val", "val_best")
print("\n=== Test split ===")
test_res, TEST_METRICS, TEST_PER_CLASS = evaluate_model(best_model, "test", "test_best")
print("\nDone. Detailed tables follow in Section 18.")

## 18. Evaluation Metrics

Headline metrics are *averaged over classes*. Precision and recall are reported by Ultralytics at the confidence threshold that maximises F1; mAP is computed over all thresholds. The per-class tables show where the model is strong or weak — classes with few instances give noisy scores, so always read the `Instances` column.

In [ ]:
summary_table = pd.DataFrame({"Validation": VAL_METRICS, "Test": TEST_METRICS}).round(4)
summary_table.index.name = "Metric"
print("Summary (all classes):")
display(summary_table)

for title, df in [("Validation", VAL_PER_CLASS), ("Test", TEST_PER_CLASS)]:
    print(f"\nPer-class metrics - {title} split")
    display(df.round(3).style.format(precision=3, na_rep="n/a").background_gradient(
        subset=["mAP50", "mAP50-95"], cmap="YlGn", vmin=0, vmax=1))

fig, ax = plt.subplots(figsize=(11, 4.5))
plot_df = TEST_PER_CLASS.melt(id_vars=["Class", "Instances"], value_vars=["Precision", "Recall", "mAP50", "mAP50-95"],
                              var_name="metric", value_name="score")
sns.barplot(data=plot_df, x="Class", y="score", hue="metric", ax=ax)
ax.set(title="Per-class metrics on the TEST split", ylim=(0, 1), xlabel="", ylabel="score")
plt.xticks(rotation=30, ha="right"); plt.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.2))
plt.tight_layout(); plt.show()

## 19. Confusion Matrix

For **object detection** the confusion matrix is built by matching predicted boxes to ground-truth boxes (IoU-based, at a fixed confidence threshold, typically 0.25):

* **Diagonal** - a predicted box with the right class matched a real object (true positive).
* **Off-diagonal cell** - the object was found but given the wrong class (e.g. a `Van` predicted as `Car`).
* **`background` column** (predicted = background) - real objects the model **missed** (false negatives).
* **`background` row** (true = background) - detections that match **no** real object (false positives).

Rows are the **true** class, columns the **predicted** class. The raw-count matrix shows absolute numbers; the normalised matrix divides each row by the number of real objects of that class (the diagonal is then the per-class recall).

In [ ]:
def plot_confusion(matrix_pred_true, class_names, normalize=False, title="", save_path=None):
    """Ultralytics stores matrix[pred, true]; we transpose so rows = TRUE class, columns = PREDICTED class."""
    labels = list(class_names) + ["background"]
    cm = np.asarray(matrix_pred_true, dtype=float).T
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums > 0)
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt=".2f" if normalize else ".0f", cmap="Blues", cbar=True,
                xticklabels=labels, yticklabels=labels, linewidths=0.5, linecolor="white", ax=ax,
                annot_kws={"size": 10}, vmin=0, vmax=1 if normalize else None)
    ax.set_xlabel("Predicted class", fontsize=12)
    ax.set_ylabel("True class", fontsize=12)
    ax.set_title(title, fontsize=14)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()


cm_test = test_res.confusion_matrix.matrix
plot_confusion(cm_test, CLASS_NAMES, normalize=False, title="Confusion matrix - TEST split (counts)",
               save_path=OUTPUT_DIR / "confusion_matrix_test_counts.png")
plot_confusion(cm_test, CLASS_NAMES, normalize=True, title="Confusion matrix - TEST split (row-normalised)",
               save_path=OUTPUT_DIR / "confusion_matrix_test_normalized.png")

## 20. Test Set Predictions

Predictions are made with `best.pt` on a **random** (seeded, not hand-picked) sample of test images, using `CONF_THRESHOLD`. The table summarises what was detected in each image.

In [ ]:
test_images = sorted((DATASET_DIR / "images" / "test").iterdir())
n_show = min(NUM_PRED_EXAMPLES, len(test_images))
sample_paths = random.Random(SEED).sample(test_images, n_show)

pred_results = best_model.predict(source=[str(p) for p in sample_paths], conf=CONF_THRESHOLD, imgsz=IMG_SIZE,
                                  device=DEVICE, verbose=False)

rows = []
for p, r in zip(sample_paths, pred_results):
    cls_ids = r.boxes.cls.cpu().numpy().astype(int) if len(r.boxes) else np.array([], dtype=int)
    cnt = Counter(CLASS_NAMES[c] for c in cls_ids)
    rows.append({"image": p.name, "detections": int(len(cls_ids)),
                 "classes": ", ".join(f"{k} x{v}" for k, v in cnt.most_common()) or "-",
                 "max_confidence": float(r.boxes.conf.max()) if len(cls_ids) else np.nan})
display(pd.DataFrame(rows).round(3))

## 21. Prediction Visualization

Each box is labelled **class + confidence** (e.g. `Car 0.91`) and coloured by class. The images shown are the random sample from Section 20 — no filtering for "good-looking" results.

In [ ]:
def result_to_items(result, names=CLASS_NAMES):
    """Convert an Ultralytics Result into draw_boxes() items and a list of detection dicts."""
    items, dets = [], []
    if result.boxes is None or len(result.boxes) == 0:
        return items, dets
    xyxy = result.boxes.xyxy.cpu().numpy()
    cls = result.boxes.cls.cpu().numpy().astype(int)
    conf = result.boxes.conf.cpu().numpy()
    for (x1, y1, x2, y2), c, s in zip(xyxy, cls, conf):
        items.append((x1, y1, x2, y2, f"{names[c]} {s:.2f}", c))
        dets.append({"class_id": int(c), "class_name": names[c], "confidence": float(s),
                     "bbox_xyxy": [float(x1), float(y1), float(x2), float(y2)]})
    return items, dets


imgs, titles = [], []
for p, r in zip(sample_paths, pred_results):
    items, dets = result_to_items(r)
    imgs.append(draw_boxes(load_image_rgb(p), items))
    titles.append(f"{p.name} - {len(dets)} detections (conf >= {CONF_THRESHOLD})")
show_grid(imgs, titles, ncols=2, size_per_image=(8, 2.9), suptitle="Test-set predictions (random sample)",
          save_path=OUTPUT_DIR / "prediction_examples.png")

## 22. Single Image Inference Function

`predict_image(image_path, model, conf=0.25)` loads an image, runs YOLO inference, draws the predictions, displays the result and returns the detections as a list of dictionaries - ready to reuse on any road image (also outside the dataset).

In [ ]:
def predict_image(image_path, model, conf=0.25, imgsz=None, show=True, figsize=(14, 5)):
    """Run inference on one image. Returns a list of dicts: class_id, class_name, confidence, bbox_xyxy."""
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")
    img_rgb = load_image_rgb(image_path)                      # raises a clear error if corrupted
    result = model.predict(source=str(image_path), conf=conf, imgsz=imgsz or IMG_SIZE, device=DEVICE,
                           verbose=False)[0]
    items, detections = result_to_items(result, names=[model.names[i] for i in range(len(model.names))])
    if show:
        plt.figure(figsize=figsize)
        plt.imshow(draw_boxes(img_rgb, items))
        plt.axis("off")
        plt.title(f"{image_path.name} - {len(detections)} detections (conf >= {conf})")
        plt.show()
    return detections


demo_candidates = [p for p in test_images if p not in sample_paths]
demo_path = random.Random(SEED + 7).choice(demo_candidates or test_images)
demo_dets = predict_image(demo_path, best_model, conf=CONF_THRESHOLD)
display(pd.DataFrame(demo_dets).round(3) if demo_dets else "No detections above the threshold.")

## 23. Save Final Model

Everything worth keeping is collected in `/kaggle/working/final_outputs/`: the trained weights (`best.pt`), `data.yaml`, the class mapping, metrics (JSON/CSV), training log and the example figures. The original Ultralytics run folder stays in `/kaggle/working/runs/`.

In [ ]:
def to_builtin(o):
    if isinstance(o, dict):
        return {k: to_builtin(v) for k, v in o.items()}
    if isinstance(o, (np.floating, float)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    return o


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST_PT, OUTPUT_DIR / "best.pt")
shutil.copy2(YAML_PATH, OUTPUT_DIR / "data.yaml")
shutil.copy2(TRAIN_DIR / "results.csv", OUTPUT_DIR / "training_results.csv")

(OUTPUT_DIR / "class_mapping.json").write_text(json.dumps(
    {"id_to_name": {i: n for i, n in enumerate(CLASS_NAMES)}, "ignored_kitti_types": sorted(IGNORED_LOWER)}, indent=2))
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(to_builtin({
    "model": MODEL_NAME, "img_size": IMG_SIZE, "epochs_configured": EPOCHS, "seed": SEED,
    "validation": VAL_METRICS, "test": TEST_METRICS}), indent=2))
VAL_PER_CLASS.to_csv(OUTPUT_DIR / "per_class_metrics_val.csv", index=False)
TEST_PER_CLASS.to_csv(OUTPUT_DIR / "per_class_metrics_test.csv", index=False)

print("Trained model (original run) :", BEST_PT)
print("Trained model (copy)         :", OUTPUT_DIR / "best.pt")
print("\nFiles in", OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name:40s} {f.stat().st_size / 1024:10.1f} KB")

print("\nReload example:\n  from ultralytics import YOLO\n  model = YOLO('/kaggle/working/final_outputs/best.pt')")

if DELETE_DATASET_COPY_AT_END:
    shutil.rmtree(DATASET_DIR)
    print("\nGenerated dataset copy deleted (DELETE_DATASET_COPY_AT_END = True).")

## 24. Final Results Summary

The summary and interpretation below are generated **from the measured values** of this run.

In [ ]:
def f3(x):
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.3f}"

epochs_run = len(df_res)
print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"Dataset            : KITTI object detection ({DATASET_ROOT.name})")
print(f"Number of classes  : {N_CLASSES} ({', '.join(CLASS_NAMES)})")
print(f"Training images    : {len(SPLITS['train'])}")
print(f"Validation images  : {len(SPLITS['val'])}")
print(f"Test images        : {len(SPLITS['test'])}")
print(f"Model              : {MODEL_NAME} ({n_params / 1e6:.2f} M parameters before head replacement)")
print(f"Epochs             : {epochs_run} run (configured {EPOCHS}, early-stop patience {PATIENCE})")
print(f"Image size         : {IMG_SIZE}")
print(f"{'':19s}{'Validation':>12s}{'Test':>12s}")
for k in ["Precision", "Recall", "mAP50", "mAP50-95"]:
    print(f"{k:19s}{f3(VAL_METRICS[k]):>12s}{f3(TEST_METRICS[k]):>12s}")
print("=" * 60)

# ---- interpretation derived only from measured numbers --------------------------------------
notes = []
notes.append(f"On the held-out test split the model reaches mAP@0.5 = {f3(TEST_METRICS['mAP50'])} and "
             f"mAP@0.5:0.95 = {f3(TEST_METRICS['mAP50-95'])} (precision {f3(TEST_METRICS['Precision'])}, "
             f"recall {f3(TEST_METRICS['Recall'])}).")
d50 = VAL_METRICS["mAP50"] - TEST_METRICS["mAP50"]
notes.append(f"Validation mAP@0.5 minus test mAP@0.5 is {d50:+.3f}; the validation split was used to pick best.pt, "
             f"and neighbouring KITTI frames can be similar across a random split, so both numbers may be optimistic.")
pc = TEST_PER_CLASS.dropna(subset=["mAP50-95"])
if len(pc):
    b, w = pc.loc[pc["mAP50-95"].idxmax()], pc.loc[pc["mAP50-95"].idxmin()]
    notes.append(f"Best class on test by mAP@0.5:0.95: {b['Class']} ({f3(b['mAP50-95'])}, {int(b['Instances'])} instances); "
                 f"weakest: {w['Class']} ({f3(w['mAP50-95'])}, {int(w['Instances'])} instances).")
few = TEST_PER_CLASS[TEST_PER_CLASS["Instances"] < 30]["Class"].tolist()
if few:
    notes.append(f"Classes with fewer than 30 test instances ({', '.join(few)}) have statistically noisy scores.")
if TEST_METRICS["Recall"] < TEST_METRICS["Precision"] - 0.10:
    notes.append("Recall is clearly lower than precision: at the evaluation threshold the model misses more objects "
                 "than it falsely reports (see the 'background' column of the confusion matrix).")
elif TEST_METRICS["Precision"] < TEST_METRICS["Recall"] - 0.10:
    notes.append("Precision is clearly lower than recall: the model reports more false detections than it misses.")
if epochs_run < EPOCHS:
    notes.append(f"Early stopping ended training after {epochs_run} of {EPOCHS} epochs.")
else:
    notes.append("Training used all configured epochs; if the validation curves were still improving at the end, "
                 "more epochs or a larger model may help.")

print("\nInterpretation (based only on the measured results):")
for n in notes:
    print(" -", n)
print("\nSaved model:", OUTPUT_DIR / "best.pt")

## 25. Conclusion

This notebook delivers a complete, reproducible detection pipeline on KITTI: automatic dataset discovery, verified KITTI→YOLO conversion with explicit class handling, a seeded 80/10/10 split, YOLOv8 fine-tuning, evaluation on validation **and** test sets (precision, recall, mAP@0.5, mAP@0.5:0.95, per-class table, confusion matrix), prediction visualisation, a reusable `predict_image()` function and saved artefacts. Quantitative conclusions are in the output of Section 24 — they depend on the actual run.

**Limitations**
* Random frame-level split of sequential driving data (possible near-duplicate frames across splits).
* `DontCare` regions are discarded, not masked, so detections of unlabelled far-away objects count as false positives.
* A single lightweight model, a single run, no hyper-parameter search; images are down-scaled to `IMG_SIZE`, which penalises small/distant objects.
* Detection only - no tracking, depth estimation or sensor fusion as a real driver-assistance stack would use.

**Ideas for improvement**
* Try `yolov8s/m.pt`, a larger `IMG_SIZE` (e.g. 960) or more epochs.
* Split by drive sequence (if sequence metadata is available) for a leakage-free estimate.
* Add class-balancing for rare classes (`Person_sitting`, `Misc`), mask `DontCare` regions in evaluation, and add object tracking across video frames for the "detection and tracking" goal of the original use case.